# Earth Engine and geemap Public Dataset Test

Use this notebook to verify that JupyterLab, geemap, and Google Earth Engine are working together. It loads a public Earth Engine dataset, displays it in geemap, creates a simple water-occurrence mask, and computes a small summary statistic over a test AOI.

In [1]:
# Required parameters
EE_PROJECT = "hardy-tenure-383607"  # Replace with a project that has Earth Engine access.

# Optional parameters
DATASET_ID = "JRC/GSW1_4/GlobalSurfaceWater"
OCCURRENCE_BAND = "occurrence"
WATER_OCCURRENCE_THRESHOLD = 50
MAP_CENTER = [-6.5, 29.6]  # Lake Tanganyika region: [lat, lon]
MAP_ZOOM = 7
REDUCE_SCALE_METERS = 1000

# Test AOI around Lake Tanganyika: [west, south, east, north]
AOI_BOUNDS = [28.5, -9.5, 31.5, -3.0]

## Imports And Earth Engine Initialization

Before running this cell, authenticate once from a terminal with `earthengine authenticate`, or run `ee.Authenticate()` manually in a notebook cell if needed.

In [2]:
import ee
import geemap

ee.Initialize(project=EE_PROJECT)
print("Earth Engine initialized")

Earth Engine initialized


## Load A Public Dataset And Build A Simple Derived Layer

This example uses the JRC Global Surface Water occurrence band. The derived layer masks pixels where historical water occurrence is at least `WATER_OCCURRENCE_THRESHOLD` percent.

In [3]:
aoi = ee.Geometry.Rectangle(AOI_BOUNDS)

surface_water = ee.Image(DATASET_ID).select(OCCURRENCE_BAND)
occurrence = surface_water.clip(aoi)
water_mask = occurrence.gte(WATER_OCCURRENCE_THRESHOLD).selfMask()

print("Dataset:", DATASET_ID)
print("Derived layer: occurrence >=", WATER_OCCURRENCE_THRESHOLD, "%")

Dataset: JRC/GSW1_4/GlobalSurfaceWater
Derived layer: occurrence >= 50 %


## Display The Dataset In geemap

If this cell shows an interactive map with the AOI, water occurrence, and derived mask layers, geemap and Earth Engine tile serving are working.

In [4]:
m = geemap.Map(center=MAP_CENTER, zoom=MAP_ZOOM, ee_initialize=False)
m.add_basemap("HYBRID")
m.addLayer(aoi, {"color": "yellow"}, "Test AOI")
m.addLayer(occurrence, {"min": 0, "max": 100, "palette": ["white", "lightblue", "blue"]}, "JRC water occurrence (%)")
m.addLayer(water_mask, {"palette": ["00ffff"]}, f"Water occurrence >= {WATER_OCCURRENCE_THRESHOLD}%")
m

Map(center=[-6.5, 29.6], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', …

## Run A Small Earth Engine Operation

This computes the mean historical water occurrence inside the test AOI. The result is intentionally small, so using `getInfo()` here is acceptable for a setup test. Avoid `getInfo()` for large collections or large outputs during real development.

In [5]:
mean_occurrence = occurrence.reduceRegion(
    reducer=ee.Reducer.mean(),
    geometry=aoi,
    scale=REDUCE_SCALE_METERS,
    maxPixels=1_000_000,
)

result = mean_occurrence.getInfo()
print("Mean water occurrence in AOI:", result)

Mean water occurrence in AOI: {'occurrence': 98.24370301791232}
